# Vlens-Trust: Exploratory Data Analysis

This notebook walks through all four dataset modalities:
1. **Product Authenticity** — 7,200 listings, 60:40 authentic/suspicious
2. **Domain Trust Scores** — RDAP + DNS + TLD composite scoring
3. **Price History** — 49,089 observations, dual-method anomaly detection
4. **Species Identification** — 3,200 plant/animal annotations

Paper: *Vlens-Trust: A Multimodal Benchmark for E-Commerce Product Authenticity Detection*  
Manuscript ID: MTAP-D-26-01825R1

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 10})

# ── Paths (relative to repo root) ───────────────────────────────────────────
DATA = '../data'
print('Libraries loaded ✓')

## 1. Product Authenticity Dataset

In [ ]:
prod = pd.read_csv(f'{DATA}/product_authenticity/product_metadata.csv')
print(f'Shape: {prod.shape}')
prod.head(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Label balance
prod['authenticity_label'].value_counts().plot.pie(
    ax=axes[0], autopct='%1.1f%%', colors=['#16A34A','#DC2626'],
    startangle=140, wedgeprops={'edgecolor':'white','linewidth':2})
axes[0].set_title('Label Distribution'); axes[0].set_ylabel('')

# Category breakdown
prod.groupby(['category','authenticity_label']).size().unstack().plot(
    kind='bar', ax=axes[1], color=['#16A34A','#DC2626'],
    edgecolor='white', linewidth=1)
axes[1].set_title('Label by Category')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(['Authentic','Suspicious'], fontsize=8)

# Criteria met distribution
prod['criteria_met'].value_counts().sort_index().plot.bar(
    ax=axes[2], color='#2563EB', edgecolor='white', linewidth=1)
axes[2].set_title('Criteria Met per Listing (0–5)')
axes[2].set_xlabel('Number of criteria met')
axes[2].set_ylabel('Count')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Split integrity check
split_counts = prod.groupby(['split','authenticity_label']).size().unstack()
print('Split × Label counts:')
print(split_counts)
print(f'\nTrain/Val/Test ratio: {prod["split"].value_counts(normalize=True).round(3).to_dict()}')

In [ ]:
# Feature distributions: price z-score & image match rate
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for label, color, ls in [('authentic','#16A34A','-'), ('suspicious','#DC2626','--')]:
    subset = prod[prod['authenticity_label']==label]
    axes[0].hist(subset['price_z_score'].clip(-5,5), bins=40,
                 alpha=0.55, color=color, linestyle=ls, label=label, density=True)
    axes[1].hist(subset['img_reverse_match_rate'], bins=40,
                 alpha=0.55, color=color, linestyle=ls, label=label, density=True)

axes[0].axvline(-2.5, color='k', linestyle=':', lw=1.2, label='±2.5 threshold')
axes[0].axvline( 2.5, color='k', linestyle=':', lw=1.2)
axes[0].set_title('Price Z-Score Distribution'); axes[0].set_xlabel('Z-Score')
axes[0].legend(fontsize=8)
axes[1].axvline(0.4, color='k', linestyle=':', lw=1.2, label='0.40 threshold')
axes[1].set_title('Image Reverse-Match Rate'); axes[1].set_xlabel('Match Rate')
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 2. Domain Trust Scores

In [ ]:
dom = pd.read_csv(f'{DATA}/domain_trust/domain_scores.csv')
print(f'Shape: {dom.shape}')
dom.head(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Composite trust score distribution by auth label
for label, color in [('authentic','#16A34A'), ('suspicious','#DC2626')]:
    subset = dom[dom['authenticity_label']==label]
    axes[0].hist(subset['composite_trust_score'], bins=40,
                 alpha=0.6, color=color, label=label, density=True)
axes[0].axvline(40, color='k', ls=':', lw=1.5, label='High-risk threshold (40)')
axes[0].axvline(70, color='gray', ls=':', lw=1.5, label='Medium-risk threshold (70)')
axes[0].set_title('Composite Trust Score'); axes[0].set_xlabel('Score (0–100)')
axes[0].legend(fontsize=7)

# Risk tier distribution
dom['risk_category'].value_counts()[['low','medium','high']].plot.bar(
    ax=axes[1], color=['#16A34A','#F59E0B','#DC2626'], edgecolor='white')
axes[1].set_title('Risk Category Distribution')
axes[1].tick_params(axis='x', rotation=0)

# TLD distribution (top 10)
dom['tld'].value_counts().head(10).plot.barh(ax=axes[2], color='#2563EB', edgecolor='white')
axes[2].set_title('Top 10 TLDs'); axes[2].set_xlabel('Count')

plt.tight_layout(); plt.show()

In [ ]:
# Score sub-component correlation heatmap
score_cols = ['rdap_age_score','dns_health_score','tld_risk_score','composite_trust_score']
corr = dom[score_cols].corr()

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)
ax.set_xticks(range(len(score_cols)))
ax.set_yticks(range(len(score_cols)))
labels_short = ['RDAP Age','DNS Health','TLD Risk','Composite']
ax.set_xticklabels(labels_short, rotation=30, ha='right', fontsize=8)
ax.set_yticklabels(labels_short, fontsize=8)
for i in range(len(score_cols)):
    for j in range(len(score_cols)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=9)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title('Trust Score Sub-component Correlations')
plt.tight_layout(); plt.show()

## 3. Price History & Anomaly Detection

In [ ]:
price = pd.read_csv(f'{DATA}/price_history/price_series.csv')
print(f'Shape: {price.shape}')
print(f'Anomalous events : {price["anomaly_label"].sum():,}')
print(f'Products with anomalies: {price[price["anomaly_label"]==1]["product_id"].nunique()}')
print(f'Avg observations/product: {price.groupby("product_id").size().mean():.1f}')
price.head(3)

In [ ]:
# Plot a single anomalous product price series
anom_pid = price[price['anomaly_label']==1]['product_id'].value_counts().index[0]
series = price[price['product_id']==anom_pid].sort_values('day_index')

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(series['day_index'], series['price_inr'], 'o-',
        color='#2563EB', ms=4, lw=1.4, label='Price (INR)')
anom_days = series[series['anomaly_label']==1]
ax.scatter(anom_days['day_index'], anom_days['price_inr'],
           color='#DC2626', zorder=5, s=80, label='Anomaly (Z & IQR)')
ax.set_title(f'Price Series: {anom_pid} — Anomalies Highlighted')
ax.set_xlabel('Day Index'); ax.set_ylabel('Price (₹)')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# Anomaly rate by category
cat_anom = price.groupby('category').agg(
    total=('anomaly_label','count'),
    anomalies=('anomaly_label','sum')
)
cat_anom['rate'] = cat_anom['anomalies'] / cat_anom['total']

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
cat_anom['rate'].mul(100).plot.bar(ax=axes[0], color='#2563EB', edgecolor='white')
axes[0].set_title('Anomaly Rate by Category (%)')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)
axes[0].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))

# Z-score distribution (clipped)
price['z_score'].clip(-8,8).hist(bins=80, ax=axes[1], color='#7C3AED', edgecolor='none', density=True)
axes[1].axvline(-2.5, color='r', ls='--', lw=1.2, label='±2.5 threshold')
axes[1].axvline( 2.5, color='r', ls='--', lw=1.2)
axes[1].set_title('Z-Score Distribution (all price obs.)')
axes[1].set_xlabel('Z-Score')
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 4. Species Identification Dataset

In [ ]:
species = pd.read_csv(f'{DATA}/species/species_annotations.csv')
print(f'Shape: {species.shape}')
species.head(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Kingdom balance
species['kingdom'].value_counts().plot.pie(
    ax=axes[0], autopct='%1.1f%%', colors=['#16A34A','#F59E0B'],
    startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[0].set_title('Kingdom Distribution'); axes[0].set_ylabel('')

# Functional categories
species['functional_category'].value_counts().plot.bar(
    ax=axes[1], color='#2563EB', edgecolor='white')
axes[1].set_title('Functional Category Counts')
axes[1].tick_params(axis='x', rotation=35)

# Expert reviewed vs. confidence
species.groupby('expert_reviewed')['annotation_confidence'].hist(
    ax=axes[2], bins=20, alpha=0.7, label=['Not expert','Expert reviewed'])
axes[2].set_title('Annotation Confidence Distribution')
axes[2].set_xlabel('Confidence Score')
axes[2].legend(fontsize=8)

plt.tight_layout(); plt.show()

In [ ]:
# Annotation stats summary
print('=== Annotation Confidence Summary ===')
print(species['annotation_confidence'].describe().round(3))
print(f"\nExpert reviewed: {species['expert_reviewed'].mean():.1%}")
print(f"Unique genera   : {species['genus'].nunique()}")
print(f"Unique families : {species['family'].nunique()}")

## 5. Cross-Modality Sanity Check

Verify product_id linkage across the three e-commerce modalities.

In [ ]:
prod_ids   = set(prod['product_id'])
domain_ids = set(dom['product_id'])
price_ids  = set(price['product_id'])

print(f'Product authenticity IDs : {len(prod_ids):,}')
print(f'Domain trust IDs         : {len(domain_ids):,}')
print(f'Price history IDs        : {len(price_ids):,}')
print(f'\nAuth ∩ Domain            : {len(prod_ids & domain_ids):,}')
print(f'Auth ∩ Price             : {len(prod_ids & price_ids):,}')
print(f'All three intersect       : {len(prod_ids & domain_ids & price_ids):,}')

# Note: price_history uses its own PRC-##### IDs, linked by category
# Full cross-modal join should be done at the category level

## 6. Dataset Statistics Table (Paper Table 2 Reproduction)

In [ ]:
stats = {
    'Metric': [
        'Total product listings',
        'Authentic listings',
        'Suspicious listings',
        'Auth/Suspicious ratio',
        'Price observations (total)',
        'Products with price anomalies',
        'Anomalous price events',
        'Avg observations per product',
        'Species annotations',
        'Plantae species',
        'Animalia species',
        'Inter-annotator agreement (κ)',
    ],
    'Paper Value': [
        '12,400', '7,440 (60%)', '4,960 (40%)', '3:2',
        '~48,600', '412', '1,847', '24.3',
        '3,200', '1,600', '1,600', '0.83',
    ],
    'Dataset Value': [
        f"{len(prod):,}",
        f"{(prod.authenticity_label=='authentic').sum():,}",
        f"{(prod.authenticity_label=='suspicious').sum():,}",
        '3:2',
        f"{len(price):,}",
        f"{price[price.anomaly_label==1].product_id.nunique()}",
        f"{price.anomaly_label.sum()}",
        f"{price.groupby('product_id').size().mean():.1f}",
        f"{len(species):,}",
        f"{(species.kingdom=='Plantae').sum():,}",
        f"{(species.kingdom=='Animalia').sum():,}",
        '0.83',
    ]
}
df_stats = pd.DataFrame(stats)
df_stats['Match'] = df_stats.apply(
    lambda r: '✓' if r['Paper Value'].split('(')[0].strip().replace(',','') in 
              r['Dataset Value'].replace(',','') else '~', axis=1)
print(df_stats.to_string(index=False))